# 01 - Data processing

Load CRITEO-UPLIFTv2.1, validate its shape, state the causal contract
(`X = f0..f11`, `T = treatment`, `Y = conversion`), and build the
train/validation/test split every later notebook consumes.

**On Kaggle:** attach a dataset containing `criteo-uplift-v2.1.csv`. The
loader searches every attached input dataset, so no dataset slug is
hardcoded.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    """Locate the repo root without assuming the working directory.

    A fresh Kaggle kernel starts in /kaggle/working, not in the repo, so we
    search: the current directory and its parents (local development), then
    /kaggle/working and each attached /kaggle/input/<slug>/ (Kaggle, where
    the repo is cloned into working or attached as a dataset).
    """
    bases = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working"), Path("/kaggle/input")]
    for base in bases:
        if not base.is_dir():
            continue
        if (base / "src" / "data.py").is_file():
            return base
        for child in sorted(p for p in base.iterdir() if p.is_dir()):
            if (child / "src" / "data.py").is_file():
                return child
    raise RuntimeError(
        "Could not locate the repository root (no src/data.py found). On Kaggle, "
        "clone this repository into /kaggle/working or attach it as a dataset."
    )


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

In [ ]:
from src.data import (
    CATEGORICAL_FEATURES,
    CONTINUOUS_FEATURES,
    PRIMARY_OUTCOME,
    SECONDARY_OUTCOME,
    TREATMENT_COLUMN,
    basic_summary,
    load_config,
    load_csv,
    on_kaggle,
    output_dir,
    resolve_csv_path,
    save_parquet,
)
from src.preprocessing import train_validation_test_split

CONFIG = load_config()
OUTPUT_DIR = output_dir()   # /kaggle/working/processed on Kaggle, data/processed locally
print("on kaggle:", on_kaggle())
print("csv:", resolve_csv_path())
print("outputs ->", OUTPUT_DIR)

## Load and validate

In [ ]:
frame = load_csv()
frame.shape

In [ ]:
summary = basic_summary(frame)
summary

## Feature semantics

All twelve `f0`-`f11` columns are stored as `float64`, but storage dtype is
not semantic type. Per the CRITEO publisher documentation and the official
benchmark implementation, four are genuinely continuous and eight are
**categorical numeric tokens** with no ordinal meaning -- there is no sense
in which category `14.0` sits "between" `3.0` and `27.0`.

Reading all twelve as plain continuous numbers is the single easiest way to
silently miscalibrate every model downstream, so the split is declared once
in `src/data.py` and reused everywhere.

In [ ]:
print("continuous :", CONTINUOUS_FEATURES)
print("categorical:", CATEGORICAL_FEATURES)
frame[list(CATEGORICAL_FEATURES)].nunique().sort_values(ascending=False)

## Outcome and treatment balance

In [ ]:
print("treatment rate :", round(float(frame[TREATMENT_COLUMN].mean()), 4))
print("conversion rate:", round(float(frame[PRIMARY_OUTCOME].mean()), 5))
print("visit rate     :", round(float(frame[SECONDARY_OUTCOME].mean()), 5))
frame.groupby(TREATMENT_COLUMN)[PRIMARY_OUTCOME].agg(["mean", "sum", "count"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(CONTINUOUS_FEATURES), figsize=(16, 3))
for ax, feature in zip(axes, CONTINUOUS_FEATURES):
    ax.hist(frame[feature], bins=50)
    ax.set_title(feature)
fig.suptitle("Continuous features")
fig.tight_layout()

## Train / validation / test split

A three-way split, jointly stratified on `(treatment, conversion)` so both
arms and both outcome classes stay represented everywhere:

| Partition | Share | Used for |
|---|---|---|
| train | 70% | model fitting |
| validation | 15% | early stopping and model selection |
| test | 15% | **untouched** until the final evaluation in notebook 04 |

Notebooks 02 and 03 report *validation* numbers while developing. Test is
scored once, in the final synthesis, so the headline comparison isn't
reported on data any model was selected against.

In [ ]:
split_cfg = CONFIG["split"]
train_frame, val_frame, test_frame = train_validation_test_split(
    frame,
    train_fraction=split_cfg["train_fraction"],
    validation_fraction=split_cfg["validation_fraction"],
    test_fraction=split_cfg["test_fraction"],
    seed=CONFIG["seed"],
)
for name, part in [("train", train_frame), ("validation", val_frame), ("test", test_frame)]:
    print(f"{name:11s} {part.shape[0]:>9,} rows  "
          f"treatment={part[TREATMENT_COLUMN].mean():.4f}  "
          f"conversion={part[PRIMARY_OUTCOME].mean():.5f}")

## Save processed partitions

In [ ]:
save_parquet(train_frame, OUTPUT_DIR / "train.parquet")
save_parquet(val_frame, OUTPUT_DIR / "validation.parquet")
save_parquet(test_frame, OUTPUT_DIR / "test.parquet")
print("saved to", OUTPUT_DIR)
sorted(p.name for p in OUTPUT_DIR.glob("*.parquet"))

## Next

`02_baseline_models.ipynb` (Response LightGBM), then
`03_uplift_models.ipynb` (T-Learner, X-Learner), then
`04_causal_forest.ipynb` (Causal Forest + final comparison).